# From v2 to v2.5: A Richer Golf-Hole Similarity Engine

**Goal of this notebook.** Show *why* the v2.5 surface-aware point-cloud model is a
meaningful improvement over the v2 engineered-feature model for finding golf holes
that **play and are designed alike** — with a real case study (Augusta National 13),
a config comparison, and a calibration health check.

> This notebook is **read-only and deterministic**. It reuses project modules
> (`pipeline.modeling.pointcloud.validate_similarity`, `pipeline.modeling.artifact_loader`)
> and the **local, gitignored** batch outputs under
> `courses/_index/pointcloud_similarity/<config>/`. If those outputs aren't present it
> degrades gracefully and tells you how to generate them — it never requires the
> internet or a Hugging Face token, and it commits no generated data.


## 1. The problem

For a given hole, we want the holes across other courses that **play similarly** —
similar corridor shape, green defense, hazard pattern, and elevation. Two models have
been built for this:

- **v2** — *engineered feature-vector* similarity. Each hole is compressed into ~90
  summary features (percentages, widths, dogleg score, elevation stats) and holes are
  compared by a weighted distance in that feature space.
- **v2.5** — *surface-aware point-cloud* similarity. Each hole keeps its actual
  normalized geometry: a tee-relative, green-aligned labeled point cloud, compared
  **per surface** (fairway / green / bunker / water / tee) with a Chamfer distance.

The rest of the notebook contrasts them and demonstrates v2.5 on real data.


In [ ]:
# --- Setup: locate the repo, import project modules, set deterministic display ---
from pathlib import Path
import sys

import numpy as np
import pandas as pd


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pipeline" / "modeling").is_dir():
            return p
    raise RuntimeError("could not locate repo root (no pipeline/modeling found)")


REPO_ROOT = _find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

from pipeline.modeling.pointcloud import validate_similarity as vsim

COURSES_ROOT = REPO_ROOT / "courses"
INDEX_ROOT = COURSES_ROOT / "_index"
PC_ROOT = INDEX_ROOT / "pointcloud_similarity"
CONFIGS = ["baseline", "hazard_heavy", "green_heavy", "fairway_heavy"]
TARGET_PC = "augusta_national:13"     # v2.5 id: course_slug:hole_number
TARGET_V2 = "augusta_national__13"    # v2 id: course_slug__NN
TOP_N = 10


def _config_dir(name):
    return PC_ROOT / name


def v25_available():
    return PC_ROOT.exists() and all(
        (_config_dir(c) / "similarity_results.csv").exists() for c in CONFIGS
    )


print("repo root:", REPO_ROOT)
print("v2.5 batch outputs present:", v25_available())
if not v25_available():
    print("  -> generate with: python -m pipeline.modeling.pointcloud.export_similarity "
          "--config configs/similarity/pointcloud_chamfer_v1.yaml --all --top-n 25")


## 2. v2 — engineered feature-vector similarity

**What it is.** Median-impute → standardize → (optionally) weight ~90 engineered
features per hole, then nearest-neighbour search in that space (cross-course, same-par,
length-guarded for the v2 table).

**Good at:** fast, broad, simple, and interpretable *at a summary-statistic level*
("these holes have similar bunker % and dogleg score").

**Limitation:** it compresses the **spatial layout** into summary features. Two holes
can share fairway %, bunker %, and length yet place those bunkers in completely
different spots and play nothing alike. The feature vector can't see *where* things are.


In [ ]:
# --- v2 top matches for Augusta National 13 (from the merged v2 table) ---
v2_csv = INDEX_ROOT / "hole_similarity_v2.csv"
if v2_csv.exists():
    v2 = pd.read_csv(v2_csv)
    keep = [c for c in ["rank", "similar_hole_id", "distance", "similar_length_m", "length_diff_m"]
            if c in v2.columns]
    v2_hole = (v2[v2["query_hole_id"] == TARGET_V2]
               .sort_values("rank").loc[:, keep].head(TOP_N).reset_index(drop=True))
    print(f"v2 top {len(v2_hole)} matches for {TARGET_V2} (lower distance = more similar):")
    display(v2_hole)
else:
    print("v2 table not found locally (courses/_index/hole_similarity_v2.csv).")
    print("Build it with: python -m pipeline.modeling all")


## 3. v2.5 — surface-aware point-cloud similarity

**What it is.** Each hole is a labeled point cloud in a **normalized frame**:

- **Tee-relative, green-aligned** — the tee is the origin and the tee→green axis points
  +Y, so every hole shares one coordinate frame and shapes are directly comparable.
- **Per-surface Chamfer distance** — fairway-to-fairway, green-to-green, etc., so the
  *spatial layout of each surface* drives the score.
- **Config-driven surface weights** — YAML presets (baseline / hazard-heavy /
  green-heavy / fairway-heavy) re-weight the surfaces to answer different questions.
- **Candidate filtering** — same par, within a yardage window, required surfaces
  present, applied **before** scoring (cheap gate, no geometry).
- **Outputs store IDs + scores only** — no raw point-cloud geometry is duplicated.

Lower `total_score` = more similar.


## 4. v2 vs v2.5 at a glance

| | **v2 (feature-vector)** | **v2.5 (point-cloud)** |
|---|---|---|
| Unit of comparison | one engineered ~90-feature row | labeled normalized point cloud |
| Distance | weighted feature-space distance | per-surface Chamfer + yardage/elevation penalties |
| Sees spatial layout? | no (summary stats only) | **yes** (where each surface actually is) |
| Tunable lens | feature weights / facets | **surface weights via YAML presets** |
| Strengths | fast, broad, simple, summary-interpretable | spatial fidelity, surface-level explainability, preset lenses |
| Best when | quick broad shortlist, light compute | design-similarity, hazard/green-specific questions, validation |


## 5. Real case study — Augusta National 13 (par 5)

We rank the field for **`augusta_national:13`** under all four v2.5 presets and look at
how stable the top matches are across weightings.


In [ ]:
# --- v2.5 top matches across all four presets (uses validate_similarity, on main) ---
def v25_top(config, target=TARGET_PC, top_n=TOP_N):
    df, _manifest, name = vsim.load_result_dir(_config_dir(config))
    return vsim.top_matches_for_target(df, target, top_n, name)


if v25_available():
    top_by_config = {c: v25_top(c) for c in CONFIGS}
    for c in CONFIGS:
        t = top_by_config[c]
        view = t[["rank", "candidate_hole_id", "total_score",
                  "fairway_score", "green_score", "bunker_score", "water_score", "tee_score"]]
        print(f"\n=== {c} - top {len(t)} for {TARGET_PC} (lower total_score = more similar) ===")
        display(view.round(3))
else:
    top_by_config = {}
    print("v2.5 outputs not present; skipping the live case study (see setup cell).")


In [ ]:
# --- Ranking stability: where does each candidate land across presets? ---
if top_by_config:
    rc = vsim.rank_comparison(top_by_config)
    rank_cols = [f"rank_{c}" for c in CONFIGS if f"rank_{c}" in rc.columns]
    stable = rc[["candidate_hole_id", *rank_cols, "configs_present_count",
                 "best_rank", "worst_rank", "rank_spread"]]
    print("Per-candidate rank across presets (NaN = not in that preset's top-N):")
    display(stable)

    core = sorted(rc.loc[rc["configs_present_count"] == len(CONFIGS), "candidate_hole_id"])
    print("\nStable all-config core set (in every preset's top-N):")
    for c in core:
        print("  -", c)
    qh = rc[rc["candidate_hole_id"] == "quail_hollow_club:15"]
    if not qh.empty:
        ranks = {c: int(qh.iloc[0][f"rank_{c}"]) for c in CONFIGS
                 if not pd.isna(qh.iloc[0].get(f"rank_{c}"))}
        print("\nQuail Hollow 15 rank by preset:", ranks)


**What to notice.**

- **Quail Hollow 15** is the **#1** match for Augusta 13 under **baseline**,
  **fairway_heavy**, and **green_heavy**, and still **#3** under **hazard_heavy**.
- A **stable core set** survives every weighting — robust plays-alike candidates:
  - Quail Hollow 15
  - Doral Blue Monster 2
  - Pebble Beach 18
  - TPC Southwind 16

That stability is the key signal: the model is responding to genuine structural
similarity, not reshuffling randomly whenever the weights change.


## 6. How much do the presets agree? (config overlap)

Jaccard overlap of the four presets' top-10 candidate sets for Augusta 13.


In [ ]:
# --- Live Jaccard overlap of the presets' top-N sets for this hole ---
if top_by_config:
    overlap = vsim.config_overlap_summary(top_by_config, TOP_N)
    display(overlap[["config_a", "config_b", "overlap_count", "union_count", "jaccard_similarity"]])
else:
    print("v2.5 outputs not present; showing the documented values instead (next cell).")

# Documented reference values (augusta_national:13, top-10) from
# docs/pointcloud_v2_5_real_findings.md:
documented_overlap = pd.DataFrame([
    ("baseline", "fairway_heavy", 0.82),
    ("baseline", "green_heavy", 0.67),
    ("fairway_heavy", "green_heavy", 0.67),
    ("baseline", "hazard_heavy", 0.33),
    ("green_heavy", "hazard_heavy", 0.33),
    ("fairway_heavy", "hazard_heavy", 0.25),
], columns=["config_a", "config_b", "documented_jaccard"])
display(documented_overlap)


**Reading it.**

- **baseline, fairway_heavy, green_heavy largely agree** (Jaccard 0.67–0.82).
- **hazard_heavy is the opinionated preset** (0.25–0.33 vs the others): re-weighting
  bunkers and water reshuffles the field the most.

That's a **feature, not a bug** — hazard_heavy is a separate "when hazards matter more"
lens, while the other presets give a stable design-similarity ranking.


## 7. Calibration health — are scores geometry-driven?

A worry with any composite score is that penalties (missing-surface, yardage, elevation)
quietly take over. The `calibrate` module decomposes every pair's score into weighted
components. Documented results across all four presets (12,055 pairs each), from
`docs/pointcloud_v2_5_real_findings.md`:


In [ ]:
# --- Documented calibration table (real run, all four presets) ---
calibration = pd.DataFrame([
    ("baseline",      12055, 0.034, 0.0003, 0.255, "none"),
    ("hazard_heavy",  12055, 0.196, 0.0003, 0.148, "none"),
    ("green_heavy",   12055, 0.009, 0.0002, 0.339, "none"),
    ("fairway_heavy", 12055, 0.001, 0.0002, 0.202, "none"),
], columns=["config", "n_pairs", "missing_dominant_frac",
            "penalty_dominant_frac", "elevation_corr", "flags"])
display(calibration)


**Reading it.**

- **Penalty dominance ≈ 0** everywhere → yardage/elevation penalties are *not* swamping
  the geometry scores.
- **Elevation double-count is not flagged** (corr 0.15–0.34, below the 0.5 threshold),
  even though `z_weight` and an explicit elevation penalty both exist.
- **Missing-surface dominance is only meaningfully high for `hazard_heavy` (19.6%)** —
  expected, since that preset weights bunker/water heavily and penalizes their absence;
  still below the 25% flag.

**Net: the scores are geometry-driven, not penalty-driven** — and no preset trips a
calibration flag.


## 8. Visual showcase — the target next to its top match

A picture of *why* two holes score as similar: the normalized surfaces of Augusta 13
beside its #1 v2.5 match (baseline preset). Rendered inline from the local point clouds;
if those aren't present, this degrades to a note.


In [ ]:
# --- Inline figure: target vs #1 baseline match, via the v2.5 visual helper ---
shown = False
if top_by_config and COURSES_ROOT.exists():
    try:
        from pipeline.modeling.pointcloud.export_similarity import CompactArtifactLoader
        from pipeline.modeling.pointcloud import visualize_similarity as vviz

        loader = CompactArtifactLoader(courses_root=COURSES_ROOT)
        target_points = loader.load_points(TARGET_PC)

        best = top_by_config["baseline"].iloc[0]
        best_id = str(best["candidate_hole_id"])
        score_row = {k: best.get(k) for k in (
            "rank", "total_score", "fairway_score", "green_score", "bunker_score",
            "water_score", "tee_score", "yardage_penalty", "elevation_penalty",
            "missing_surface_penalty")}
        cand_points = loader.load_points(best_id)

        if target_points and cand_points:
            import io
            import matplotlib.pyplot as plt
            from IPython.display import Image as _Image, display as _display

            fig = vviz.build_comparison_figure(
                TARGET_PC, target_points, [(best_id, cand_points, score_row)])
            # visualize_similarity forces the Agg backend, so render to PNG bytes
            # explicitly rather than relying on the inline display hook.
            buf = io.BytesIO()
            fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
            plt.close(fig)
            _display(_Image(data=buf.getvalue()))
            shown = True
            print(f"Surfaces: {TARGET_PC} (target) vs {best_id} (#1 baseline match).")
    except Exception as exc:  # visual is best-effort; never break the narrative
        print("Visual unavailable:", type(exc).__name__, exc)

if not shown:
    print("Point-cloud visual skipped (local compact clouds under courses/ not present).")
    print("It renders automatically when the courses/ point clouds and v2.5 outputs exist.")


## 9. Conclusion

- **v2 is a useful, fast baseline** — broad, simple, and interpretable at the
  summary-statistic level. Good for a quick shortlist.
- **v2.5 is a richer design-similarity engine** — it compares the actual normalized
  surface geometry, so it captures *where* fairway, green, and hazards sit, not just how
  much of each there is.
- **v2.5 improves interpretability** (per-surface score breakdown), **spatial fidelity**
  (Chamfer over real layouts), and **model validation** (ranking stability across presets
  + a clean calibration profile).
- **Honest caveat:** this demonstrates **model behavior and ranking stability**, not
  player-performance prediction. A formal **human-rated plays-alike benchmark** is still
  needed before claiming a numeric "% improvement." The robust core set for Augusta 13
  (Quail Hollow 15, Doral Blue Monster 2, Pebble Beach 18, TPC Southwind 16) is strong
  qualitative evidence pointing that way.
